# Jour 1 · Construire un premier RAG de bout en bout


## Ce que nous ajoutons

Le notebook précédent s'arrêtait aux documents retrouvés. Nous ajoutons la génération :

```text
documents retrouvés + question → prompt → LLM → réponse
```

Nous gardons d'abord chaque variable afin de pouvoir l'afficher et la comprendre.


![Flux complet d'un premier RAG](../assets/jour_01/02_flux_rag_decompose.png)

*La fonction `format_docs` est le pont entre les objets `Document` et le texte attendu dans le prompt.*


In [1]:
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Racine du cours :", ROOT)


Racine du cours : C:\Users\bstorm_user\Desktop\rag\cours_rag


In [2]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


Modèle de génération : qwen2.5:3b
Modèle d'embeddings  : bge-m3


In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "Pour un écran noir, vérifier le voyant du moniteur, le câble d'alimentation, "
            "le câble HDMI ou DisplayPort et la source vidéo sélectionnée."
        ),
        metadata={"source": "procedure_ecran.md", "categorie": "materiel"},
    ),
    Document(
        page_content=(
            "Pour se connecter à distance, ouvrir GlobalProtect, saisir le portail "
            "vpn.entreprise.test puis valider le code MFA."
        ),
        metadata={"source": "procedure_vpn.md", "categorie": "reseau"},
    ),
    Document(
        page_content=(
            "En cas de bourrage, éteindre l'imprimante, retirer doucement le papier "
            "dans le sens de circulation puis contrôler les rouleaux."
        ),
        metadata={"source": "procedure_imprimante.md", "categorie": "materiel"},
    ),
    Document(
        page_content=(
            "Un technicien ne demande jamais un mot de passe ni un code MFA. "
            "Ces secrets ne doivent pas être placés dans un ticket."
        ),
        metadata={"source": "politique_securite.md", "categorie": "securite"},
    ),
]

print(f"{len(documents)} documents préparés")


4 documents préparés


In [4]:
import chromadb
from langchain_chroma import Chroma

client = chromadb.EphemeralClient()
vectorstore = Chroma(
    client=client,
    collection_name="helpdesk_demo",
    embedding_function=embeddings,
)
vectorstore.add_documents(
    documents,
    ids=[f"doc-{index}" for index in range(len(documents))],
)

print("Documents indexés :", len(vectorstore.get()["ids"]))


Documents indexés : 4


## Étape 1 — récupérer les documents

Aucun LLM n'est encore appelé. Nous contrôlons les sources avant de poursuivre.


In [5]:
question = "Je travaille à distance. Quel portail dois-je saisir dans GlobalProtect ?"
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
documents_retrouves = retriever.invoke(question)

for document in documents_retrouves:
    print(document.metadata["source"], "→", document.page_content)


procedure_vpn.md → Pour se connecter à distance, ouvrir GlobalProtect, saisir le portail vpn.entreprise.test puis valider le code MFA.
politique_securite.md → Un technicien ne demande jamais un mot de passe ni un code MFA. Ces secrets ne doivent pas être placés dans un ticket.


## Étape 2 — fabriquer le contexte

Le retriever renvoie une `list[Document]`. Un template attend du texte. `format_docs` extrait chaque `page_content` et les relie.

Nous indiquons aussi la source devant chaque passage pour rendre le prompt lisible. Les citations finales seront construites depuis les métadonnées réelles.


### Pourquoi faut-il transformer les documents ?

Le retriever renvoie une liste de boîtes `Document`. Chaque boîte contient du texte dans `page_content` et des étiquettes dans `metadata`.

Le prompt, lui, attend une simple chaîne de caractères. `format_docs` ouvre donc les boîtes, récupère leur texte et rassemble les passages. C'est un changement de format, pas une nouvelle recherche.


In [6]:
def format_docs(docs):
    passages = []
    for doc in docs:
        source = doc.metadata.get("source", "source inconnue")
        passages.append(f"[Source : {source}]\n{doc.page_content}")
    return "\n\n".join(passages)


contexte = format_docs(documents_retrouves)
print(contexte)


[Source : procedure_vpn.md]
Pour se connecter à distance, ouvrir GlobalProtect, saisir le portail vpn.entreprise.test puis valider le code MFA.

[Source : politique_securite.md]
Un technicien ne demande jamais un mot de passe ni un code MFA. Ces secrets ne doivent pas être placés dans un ticket.


### À vous de jouer — réécrire `format_docs`

Écrivez une version courte qui relie uniquement les `page_content` par deux sauts de ligne.


In [7]:
# À vous de jouer — remplacez pass par la jointure des page_content.
def format_docs_exercice(docs):
    pass


print("Fonction créée. Comparez ensuite avec format_docs.")


Fonction créée. Comparez ensuite avec format_docs.


## Étape 3 — construire le prompt

Le **contexte** apporte les informations ; la **question** indique ce qu'il faut en faire. Le prompt ajoute une règle de refus lorsque les passages ne suffisent pas.


### Le prompt est le dossier remis au modèle

Le **template** est un texte à trous. À l'exécution, `{contexte}` reçoit les passages retrouvés et `{question}` reçoit la demande de l'utilisateur.

Afficher le prompt rempli avant l'appel du LLM permet de vérifier simplement ce que le modèle va réellement lire. Si une information n'apparaît pas ici, le modèle ne doit pas être chargé de la deviner.


![Contexte et question réunis dans le prompt](../assets/jour_01/02_contexte_et_question.png)

*Le modèle reçoit les règles, les passages retrouvés et la question dans un même ensemble de messages.*


In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt_rag = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu es un assistant helpdesk. Réponds uniquement avec le CONTEXTE. "
        "Si le contexte ne permet pas de répondre, écris exactement : "
        "Je ne trouve pas cette information dans la documentation fournie.\n\n"
        "CONTEXTE :\n{contexte}",
    ),
    ("human", "QUESTION : {question}"),
])

prompt_prepare = prompt_rag.invoke({"contexte": contexte, "question": question})
for message in prompt_prepare.to_messages():
    print(f"\n--- {message.type.upper()} ---")
    print(message.content)



--- SYSTEM ---
Tu es un assistant helpdesk. Réponds uniquement avec le CONTEXTE. Si le contexte ne permet pas de répondre, écris exactement : Je ne trouve pas cette information dans la documentation fournie.

CONTEXTE :
[Source : procedure_vpn.md]
Pour se connecter à distance, ouvrir GlobalProtect, saisir le portail vpn.entreprise.test puis valider le code MFA.

[Source : politique_securite.md]
Un technicien ne demande jamais un mot de passe ni un code MFA. Ces secrets ne doivent pas être placés dans un ticket.

--- HUMAN ---
QUESTION : Je travaille à distance. Quel portail dois-je saisir dans GlobalProtect ?


## Étape 4 — appeler le modèle

`StrOutputParser` transforme l'objet `AIMessage` en chaîne Python. Le modèle reçoit exactement le prompt affiché ci-dessus.


In [9]:
from langchain_core.output_parsers import StrOutputParser

chaine_generation = prompt_rag | llm | StrOutputParser()
reponse = chaine_generation.invoke({
    "contexte": contexte,
    "question": question,
})
print(reponse)


Pour se connecter à distance, vous devez saisir le portail vpn.vpn.vpn.vpntentreprise.test dans GlobalProtect.


## Étape 5 — afficher les vraies sources

Nous ne demandons pas au modèle d'inventer une bibliographie. Les sources viennent des documents réellement remontés par le retriever.


In [10]:
sources = sorted({doc.metadata["source"] for doc in documents_retrouves})
print("RÉPONSE")
print(reponse)
print("\nSOURCES")
for source in sources:
    print("-", source)


RÉPONSE
Pour se connecter à distance, vous devez saisir le portail vpn.vpn.vpn.vpntentreprise.test dans GlobalProtect.

SOURCES
- politique_securite.md
- procedure_vpn.md


## La même chaîne, maintenant en LCEL

**LCEL** signifie *LangChain Expression Language*. Ce n'est pas un nouveau raisonnement : c'est une écriture permettant de relier des composants avec l'opérateur `|`.

Le dictionnaire ci-dessous est automatiquement converti en **`RunnableParallel`**. Un `Runnable` est simplement un composant que LangChain sait appeler avec `.invoke()`.

- la branche `contexte` lance le retriever puis `format_docs` ;
- la branche `question` conserve la question originale avec `RunnablePassthrough` ;
- les deux résultats sont réunis avant d'entrer dans le prompt.

`Parallel` décrit donc deux transformations partant de la même entrée. Cela ne change ni l'ordre logique du RAG ni ce que reçoit le modèle.


In [11]:
from langchain_core.runnables import RunnablePassthrough

chaine_rag = (
    {
        "contexte": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

print(chaine_rag.invoke(question))


Pour se connecter à distance, vous devez saisir le portail vpn.vpn.vpn.vpntentreprise.test dans GlobalProtect.


## Pourquoi inspecter la recherche en premier ?

L'ordre de diagnostic suit le flux : question, documents, contexte, prompt, réponse.


![Conséquence d'un bon ou mauvais document retrouvé](../assets/jour_01/02_bon_et_mauvais_retrieval.png)

*Le LLM ne doit pas compenser une connaissance absente ; il doit refuser lorsque le contexte est insuffisant.*


In [12]:
question_hors_perimetre = "Quelle sera la météo demain ?"
docs_hors_perimetre = retriever.invoke(question_hors_perimetre)

print("DOCUMENTS LES PLUS PROCHES, MAIS NON PERTINENTS :")
for document in docs_hors_perimetre:
    print("-", document.metadata["source"])

print("\nRÉPONSE :")
print(chaine_rag.invoke(question_hors_perimetre))


DOCUMENTS LES PLUS PROCHES, MAIS NON PERTINENTS :
- politique_securite.md
- procedure_imprimante.md

RÉPONSE :
Je ne trouve pas cette information dans la documentation fournie.


## À retenir

- Le retriever produit des objets `Document`.
- `format_docs` produit le texte du contexte.
- Le prompt combine règles, contexte et question.
- Les sources sont extraites des métadonnées réelles.
- La version LCEL compacte vient après la version observable.
